# BC–WA PM2.5 benchmark on Kaggle

**Settings (right panel):** Accelerator **GPU T4 x2** · Internet **On** (needs a phone-verified account).
Then **Save Version → Save & Run All** - it runs in the background (browser can be closed) for up to 12 h,
and everything in `/kaggle/working/metrics` is saved as this version's output.

One version = one model group. To continue a group, or to combine groups, add earlier versions'
output as input (**Add Input → Notebooks → this notebook → that version**); their score files are
picked up automatically.

In [ ]:
GROUP = 1          # 1: MLP GRU LSTM PM25_GNN AirLapseV2 Crossformer AirPhyNet Transformer AirDualODE
                   # 2: STMamba Informer AirFormer    3: Autoformer TCN_DIR    4: AirDDE
REPEATS = 5
MAX_HOURS = 9      # stop starting new runs after this; leaves time for the last run under Kaggle's 12 h

In [ ]:
import glob, os, shutil, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Accelerator to GPU T4 x2')

!git clone -q https://github.com/Grace-VN/BC-WA-PM2.5-Forecast.git /tmp/repo
%cd /tmp/repo
!pip install -q -r requirements.txt

METRICS = '/kaggle/working/metrics'          # saved as notebook output
os.makedirs(f'{METRICS}/scores', exist_ok=True)
os.environ['METRICS_DIR'] = METRICS
os.environ['RESULTS_DIR'] = '/tmp/results'   # large raw arrays: not saved

# resume / combine: copy score files from any attached earlier outputs
found = glob.glob('/kaggle/input/**/scores/*.csv', recursive=True)
for f in found:
    shutil.copy(f, f'{METRICS}/scores/')
print(f'{len(found)} earlier run results picked up')

In [ ]:
!python benchmark.py --group {GROUP} --repeats {REPEATS} --epochs 50 --early_stop 10 --max_hours {MAX_HOURS}

In [ ]:
# all results available in this version (this group + any attached outputs): mean ± std
!python benchmark.py --summary
import pandas as pd
s = pd.read_csv(f'{METRICS}/benchmark_summary.csv', index_col=0)
cols = ['RMSE', 'MAE', 'MAPE', 'RMSE_gt35.5', 'CSI@35.5', 'POD@35.5', 'FAR@35.5', 'CSI@55.5']
table = pd.DataFrame({'runs': s.n_repeats})
for c in cols:
    d = 1 if c == 'MAPE' else 3
    table[c] = [f'{m:.{d}f} ± {sd:.{d}f}' if pd.notna(sd) else f'{m:.{d}f}'
                for m, sd in zip(s[c + '_mean'], s[c + '_std'])]
table